In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import platform

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent


RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ibm_tabformer"
).resolve()


ARCHIVE_PATH = RAW_DIR / "transactions.tgz"

CSV_PATH = RAW_DIR / "card_transaction.v1.csv"


print("Project root:", PROJECT_ROOT)
print("Raw directory:", RAW_DIR)

print("\nArchive exists:", ARCHIVE_PATH.is_file())
print("CSV exists:", CSV_PATH.is_file())

print("\nPython:", platform.python_version())
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Project root: /Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening
Raw directory: /Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/raw/ibm_tabformer

Archive exists: True
CSV exists: True

Python: 3.14.6
Pandas: 3.0.5
NumPy: 2.5.3


In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()


def print_file_record(path):
    if not path.is_file():
        print(f"Không tìm thấy: {path}")
        return

    print("File:", path.name)
    print("Size (bytes):", path.stat().st_size)
    print(
        "Size (MiB):",
        round(path.stat().st_size / 1024**2, 2)
    )
    print("SHA-256:", sha256_file(path))


print("=== ARCHIVE ===")
print_file_record(ARCHIVE_PATH)

print("\n=== EXTRACTED CSV ===")
print_file_record(CSV_PATH)

=== ARCHIVE ===
File: transactions.tgz
Size (bytes): 278576638
Size (MiB): 265.67
SHA-256: e9f589a0958f40d60f81b1a2e8428db86e00c05755caf44fb055827976c0efa2

=== EXTRACTED CSV ===
File: card_transaction.v1.csv
Size (bytes): 2354626737
Size (MiB): 2245.55
SHA-256: 68c438319cf27614d5564b7b520814036f0d7e087b3a17d8c15081848e0f02de


In [3]:
preview = pd.read_csv(
    CSV_PATH,
    nrows=10_000
)

print("Preview shape:", preview.shape)

print("\nColumns:")
print(preview.columns.tolist())

print("\nDtypes:")
print(preview.dtypes.to_string())

print("\nFirst 5 rows:")
print(preview.head().to_string(index=False))

print("\nRandom 5 rows from preview:")
print(
    preview.sample(
        n=5,
        random_state=42
    ).to_string(index=False)
)

Preview shape: (10000, 15)

Columns:
['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']

Dtypes:
User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time                  str
Amount                str
Use Chip              str
Merchant Name       int64
Merchant City         str
Merchant State        str
Zip               float64
MCC                 int64
Errors?               str
Is Fraud?             str

First 5 rows:
 User  Card  Year  Month  Day  Time  Amount          Use Chip       Merchant Name Merchant City Merchant State     Zip  MCC Errors? Is Fraud?
    0     0  2002      9    1 06:21 $134.09 Swipe Transaction 3527213246127876953      La Verne             CA 91750.0 5300     NaN        No
    0     0  2002      9    1 06:42  $38.48 Swipe Transaction -727612092139916043 Monterey Park         

In [4]:
expected_columns = [
    "User",
    "Card",
    "Year",
    "Month",
    "Day",
    "Time",
    "Amount",
    "Use Chip",
    "Merchant Name",
    "Merchant City",
    "Merchant State",
    "Zip",
    "MCC",
    "Errors?",
    "Is Fraud?",
]


actual_columns = preview.columns.tolist()


print("Expected column count:", len(expected_columns))
print("Actual preview column count:", preview.shape[1])

print(
    "Missing columns:",
    sorted(set(expected_columns) - set(actual_columns))
)

print(
    "Unexpected columns:",
    sorted(set(actual_columns) - set(expected_columns))
)

print(
    "Column order matches:",
    actual_columns == expected_columns
)

print(
    "Duplicate column names:",
    int(preview.columns.duplicated().sum())
)

Expected column count: 15
Actual preview column count: 15
Missing columns: []
Unexpected columns: []
Column order matches: True
Duplicate column names: 0


In [5]:
print("Amount examples:")
print(
    preview["Amount"]
    .dropna()
    .head(10)
    .tolist()
)

print("\nTime examples:")
print(
    preview["Time"]
    .dropna()
    .head(10)
    .tolist()
)

print("\nUse Chip values:")
print(
    preview["Use Chip"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nTarget values:")
print(
    preview["Is Fraud?"]
    .value_counts(dropna=False)
    .to_string()
)

Amount examples:
['$134.09', '$38.48', '$120.34', '$128.95', '$104.71', '$86.19', '$93.84', '$123.50', '$61.72', '$57.10']

Time examples:
['06:21', '06:42', '06:22', '17:45', '06:23', '13:53', '05:51', '06:09', '06:14', '09:35']

Use Chip values:
Use Chip
Swipe Transaction     7108
Chip Transaction      2310
Online Transaction     582

Target values:
Is Fraud?
No     9985
Yes      15


In [6]:
CHUNK_SIZE = 500_000

row_count = 0

missing_counts = None

target_counts = Counter()
use_chip_counts = Counter()
error_counts = Counter()
year_counts = Counter()

user_values = set()
user_card_pairs = set()

merchant_values = set()
merchant_city_values = set()
merchant_state_values = set()
zip_values = set()
mcc_values = set()
time_values = set()

year_min = None
year_max = None

month_min = None
month_max = None

day_min = None
day_max = None

amount_parse_fail_count = 0
amount_negative_count = 0
amount_zero_count = 0
amount_min = None
amount_max = None

invalid_time_count = 0

chunk_number = 0


for chunk in pd.read_csv(
    CSV_PATH,
    chunksize=CHUNK_SIZE
):
    chunk_number += 1

    rows = len(chunk)
    row_count += rows

    # -----------------------
    # Missing
    # -----------------------
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing.copy()
    else:
        missing_counts = missing_counts.add(
            current_missing,
            fill_value=0
        )

    # -----------------------
    # Target
    # -----------------------
    target_counts.update(
        chunk["Is Fraud?"]
        .fillna("<MISSING>")
        .astype(str)
        .value_counts()
        .to_dict()
    )

    # -----------------------
    # Use Chip
    # -----------------------
    use_chip_counts.update(
        chunk["Use Chip"]
        .fillna("<MISSING>")
        .astype(str)
        .value_counts()
        .to_dict()
    )

    # -----------------------
    # Errors
    # -----------------------
    error_counts.update(
        chunk["Errors?"]
        .fillna("<MISSING>")
        .astype(str)
        .value_counts()
        .to_dict()
    )

    # -----------------------
    # Year
    # -----------------------
    year_counts.update(
        chunk["Year"]
        .value_counts(dropna=False)
        .to_dict()
    )

    chunk_year_min = chunk["Year"].min()
    chunk_year_max = chunk["Year"].max()

    year_min = (
        chunk_year_min
        if year_min is None
        else min(year_min, chunk_year_min)
    )

    year_max = (
        chunk_year_max
        if year_max is None
        else max(year_max, chunk_year_max)
    )

    # -----------------------
    # Month / Day
    # -----------------------
    cmn = chunk["Month"].min()
    cmx = chunk["Month"].max()

    dmn = chunk["Day"].min()
    dmx = chunk["Day"].max()

    month_min = cmn if month_min is None else min(month_min, cmn)
    month_max = cmx if month_max is None else max(month_max, cmx)

    day_min = dmn if day_min is None else min(day_min, dmn)
    day_max = dmx if day_max is None else max(day_max, dmx)

    # -----------------------
    # Cardinality / identifiers
    # -----------------------
    user_values.update(
        chunk["User"].dropna().unique().tolist()
    )

    user_card_pairs.update(
        zip(
            chunk["User"].tolist(),
            chunk["Card"].tolist()
        )
    )

    merchant_values.update(
        chunk["Merchant Name"]
        .dropna()
        .unique()
        .tolist()
    )

    merchant_city_values.update(
        chunk["Merchant City"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    merchant_state_values.update(
        chunk["Merchant State"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    zip_values.update(
        chunk["Zip"]
        .dropna()
        .unique()
        .tolist()
    )

    mcc_values.update(
        chunk["MCC"]
        .dropna()
        .unique()
        .tolist()
    )

    time_values.update(
        chunk["Time"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    # -----------------------
    # Amount: chỉ parse để audit
    # KHÔNG ghi đè raw column
    # -----------------------
    amount_numeric = pd.to_numeric(
        chunk["Amount"]
        .astype("string")
        .str.replace("$", "", regex=False),
        errors="coerce"
    )

    parse_fail_mask = (
        chunk["Amount"].notna()
        & amount_numeric.isna()
    )

    amount_parse_fail_count += int(
        parse_fail_mask.sum()
    )

    amount_negative_count += int(
        (amount_numeric < 0).sum()
    )

    amount_zero_count += int(
        (amount_numeric == 0).sum()
    )

    local_min = amount_numeric.min()
    local_max = amount_numeric.max()

    if pd.notna(local_min):
        amount_min = (
            local_min
            if amount_min is None
            else min(amount_min, local_min)
        )

    if pd.notna(local_max):
        amount_max = (
            local_max
            if amount_max is None
            else max(amount_max, local_max)
        )

    # -----------------------
    # Time format HH:MM
    # -----------------------
    time_text = chunk["Time"].astype("string")

    parts = time_text.str.extract(
        r"^(\d{1,2}):(\d{2})$"
    )

    hours = pd.to_numeric(
        parts[0],
        errors="coerce"
    )

    minutes = pd.to_numeric(
        parts[1],
        errors="coerce"
    )

    invalid_time = (
        time_text.notna()
        & (
            hours.isna()
            | minutes.isna()
            | (hours < 0)
            | (hours > 23)
            | (minutes < 0)
            | (minutes > 59)
        )
    )

    invalid_time_count += int(
        invalid_time.sum()
    )

    print(
        f"Processed chunk {chunk_number}: "
        f"{row_count:,} rows"
    )


print("\nFULL SCAN FINISHED")

Processed chunk 1: 500,000 rows
Processed chunk 2: 1,000,000 rows
Processed chunk 3: 1,500,000 rows
Processed chunk 4: 2,000,000 rows
Processed chunk 5: 2,500,000 rows
Processed chunk 6: 3,000,000 rows
Processed chunk 7: 3,500,000 rows
Processed chunk 8: 4,000,000 rows
Processed chunk 9: 4,500,000 rows
Processed chunk 10: 5,000,000 rows
Processed chunk 11: 5,500,000 rows
Processed chunk 12: 6,000,000 rows
Processed chunk 13: 6,500,000 rows
Processed chunk 14: 7,000,000 rows
Processed chunk 15: 7,500,000 rows
Processed chunk 16: 8,000,000 rows
Processed chunk 17: 8,500,000 rows
Processed chunk 18: 9,000,000 rows
Processed chunk 19: 9,500,000 rows
Processed chunk 20: 10,000,000 rows
Processed chunk 21: 10,500,000 rows
Processed chunk 22: 11,000,000 rows
Processed chunk 23: 11,500,000 rows
Processed chunk 24: 12,000,000 rows
Processed chunk 25: 12,500,000 rows
Processed chunk 26: 13,000,000 rows
Processed chunk 27: 13,500,000 rows
Processed chunk 28: 14,000,000 rows
Processed chunk 29: 14

In [7]:
print("=== BASIC SHAPE ===")
print("Total rows:", f"{row_count:,}")
print("Expected columns:", len(expected_columns))


print("\n=== MISSING COUNTS ===")
print(
    missing_counts
    .astype("int64")
    .to_string()
)

print(
    "\nTotal missing cells:",
    int(missing_counts.sum())
)


print("\n=== IDENTIFIERS / CARDINALITY ===")

print(
    "Unique Users:",
    len(user_values)
)

print(
    "Unique User+Card pairs:",
    len(user_card_pairs)
)

print(
    "Unique Merchant Name:",
    len(merchant_values)
)

print(
    "Unique Merchant City:",
    len(merchant_city_values)
)

print(
    "Unique Merchant State:",
    len(merchant_state_values)
)

print(
    "Unique Zip:",
    len(zip_values)
)

print(
    "Unique MCC:",
    len(mcc_values)
)

print(
    "Unique Time values:",
    len(time_values)
)


print("\n=== TEMPORAL ===")

print("Year min:", year_min)
print("Year max:", year_max)

print("Month min:", month_min)
print("Month max:", month_max)

print("Day min:", day_min)
print("Day max:", day_max)

print("\nRows by year:")

for year in sorted(year_counts):
    print(
        year,
        f"{year_counts[year]:,}"
    )


print("\n=== AMOUNT TECHNICAL CHECK ===")

print(
    "Amount parse failures:",
    amount_parse_fail_count
)

print(
    "Negative amounts:",
    amount_negative_count
)

print(
    "Zero amounts:",
    amount_zero_count
)

print("Amount min:", amount_min)
print("Amount max:", amount_max)


print("\n=== TIME TECHNICAL CHECK ===")

print(
    "Invalid Time values:",
    invalid_time_count
)

=== BASIC SHAPE ===
Total rows: 24,386,900
Expected columns: 15

=== MISSING COUNTS ===
User                     0
Card                     0
Year                     0
Month                    0
Day                      0
Time                     0
Amount                   0
Use Chip                 0
Merchant Name            0
Merchant City            0
Merchant State     2720821
Zip                2878135
MCC                      0
Errors?           23998469
Is Fraud?                0

Total missing cells: 29597425

=== IDENTIFIERS / CARDINALITY ===
Unique Users: 2000
Unique User+Card pairs: 6139
Unique Merchant Name: 100343
Unique Merchant City: 13429
Unique Merchant State: 223
Unique Zip: 27321
Unique MCC: 109
Unique Time values: 1440

=== TEMPORAL ===
Year min: 1991
Year max: 2020
Month min: 1
Month max: 12
Day min: 1
Day max: 31

Rows by year:
1991 1,585
1992 5,134
1993 8,378
1994 14,316
1995 20,928
1996 29,945
1997 49,753
1998 78,345
1999 118,250
2000 177,729
2001 257,998
2002 

In [8]:
print("=== Is Fraud? ===")

total_target = sum(
    target_counts.values()
)

for value, count in target_counts.most_common():
    pct = count / total_target * 100

    print(
        repr(value),
        f"{count:,}",
        f"{pct:.6f}%"
    )


print(
    "\nUnique target values:",
    sorted(
        target_counts.keys()
    )
)

=== Is Fraud? ===
'No' 24,357,143 99.877980%
'Yes' 29,757 0.122020%

Unique target values: ['No', 'Yes']


In [9]:
print("=== Use Chip ===")

for value, count in use_chip_counts.most_common():
    print(
        repr(value),
        f"{count:,}"
    )

print(
    "\nUnique Use Chip values:",
    len(use_chip_counts)
)

=== Use Chip ===
'Swipe Transaction' 15,386,082
'Chip Transaction' 6,287,598
'Online Transaction' 2,713,220

Unique Use Chip values: 3


In [10]:
print("=== Errors? ===")

for value, count in error_counts.most_common():
    print(
        repr(value),
        f"{count:,}"
    )

print(
    "\nUnique Errors? values:",
    len(error_counts)
)

=== Errors? ===
'<MISSING>' 23,998,469
'Insufficient Balance,' 242,783
'Bad PIN,' 58,918
'Technical Glitch,' 48,157
'Bad Card Number,' 13,321
'Bad CVV,' 10,740
'Bad Expiration,' 10,716
'Bad Zipcode,' 2,079
'Bad PIN,Insufficient Balance,' 581
'Insufficient Balance,Technical Glitch,' 457
'Bad PIN,Technical Glitch,' 128
'Bad Card Number,Insufficient Balance,' 122
'Bad CVV,Insufficient Balance,' 89
'Bad Expiration,Insufficient Balance,' 78
'Bad Card Number,Bad CVV,' 60
'Bad Card Number,Bad Expiration,' 54
'Bad Expiration,Bad CVV,' 47
'Bad Expiration,Technical Glitch,' 32
'Bad Card Number,Technical Glitch,' 25
'Bad CVV,Technical Glitch,' 21
'Bad Zipcode,Insufficient Balance,' 13
'Bad Zipcode,Technical Glitch,' 7
'Bad Card Number,Bad Expiration,Insufficient Balance,' 2
'Bad Card Number,Bad Expiration,Technical Glitch,' 1

Unique Errors? values: 24


In [11]:
first_chunk = next(
    pd.read_csv(
        CSV_PATH,
        chunksize=500_000
    )
)

print(
    first_chunk.dtypes.to_string()
)

print("\nDataFrame info:")
first_chunk.info(
    memory_usage="deep"
)

User                int64
Card                int64
Year                int64
Month               int64
Day                 int64
Time                  str
Amount                str
Use Chip              str
Merchant Name       int64
Merchant City         str
Merchant State        str
Zip               float64
MCC                 int64
Errors?               str
Is Fraud?             str

DataFrame info:
<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   User            500000 non-null  int64  
 1   Card            500000 non-null  int64  
 2   Year            500000 non-null  int64  
 3   Month           500000 non-null  int64  
 4   Day             500000 non-null  int64  
 5   Time            500000 non-null  str    
 6   Amount          500000 non-null  str    
 7   Use Chip        500000 non-null  str    
 8   Merchant Name   500000 non-n

In [12]:
invalid_month_count = 0
invalid_day_count = 0

for chunk in pd.read_csv(
    CSV_PATH,
    usecols=[
        "Month",
        "Day",
    ],
    chunksize=500_000
):
    invalid_month_count += int(
        (
            (chunk["Month"] < 1)
            | (chunk["Month"] > 12)
        ).sum()
    )

    invalid_day_count += int(
        (
            (chunk["Day"] < 1)
            | (chunk["Day"] > 31)
        ).sum()
    )


print(
    "Invalid Month values:",
    invalid_month_count
)

print(
    "Invalid Day values (basic 1..31 check):",
    invalid_day_count
)

print(
    "Invalid Time values:",
    invalid_time_count
)

Invalid Month values: 0
Invalid Day values (basic 1..31 check): 0
Invalid Time values: 0


In [13]:
df = pd.read_csv(CSV_PATH)

print(
    "Full dataframe shape:",
    df.shape
)

duplicate_extra = int(
    df.duplicated().sum()
)

duplicate_members = int(
    df.duplicated(
        keep=False
    ).sum()
)

print(
    "Exact duplicate extra rows:",
    duplicate_extra
)

print(
    "Rows belonging to duplicate groups:",
    duplicate_members
)

Full dataframe shape: (24386900, 15)
Exact duplicate extra rows: 66
Rows belonging to duplicate groups: 132
